In [1]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/nassau_candy_enriched.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'])

print(f"Enriched data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Enriched data loaded: 6013 rows, 28 columns


### 1. Division Summary

In [3]:
total_revenue = df['Sales'].sum()
total_profit  = df['Gross Profit'].sum()

division_summary = df.groupby('Division').agg(
    Total_Revenue   = ('Sales', 'sum'),
    Total_Profit    = ('Gross Profit', 'sum'),
    Total_Cost      = ('Cost', 'sum'),
    Total_Units     = ('Units', 'sum'),
    Total_Orders    = ('Order ID', 'nunique'),
    Num_Products    = ('Product Name', 'nunique'),
    Avg_Margin      = ('Gross_Margin_%', 'mean'),
    Avg_Profit_per_Unit = ('Profit_per_Unit', 'mean')
).round(2).reset_index()

# Profit efficiency — how much profit generated per dollar of revenue
division_summary['Profit_Efficiency'] = (
    division_summary['Total_Profit'] / division_summary['Total_Revenue']
).round(4)

division_summary['Revenue_Share_%'] = (
    division_summary['Total_Revenue'] / total_revenue * 100
).round(2)

division_summary['Profit_Share_%'] = (
    division_summary['Total_Profit'] / total_profit * 100
).round(2)

division_summary = division_summary.sort_values('Total_Profit', ascending=False).reset_index(drop=True)

print("DIVISION SUMMARY")
print(division_summary.to_string(index=False))

DIVISION SUMMARY
 Division  Total_Revenue  Total_Profit  Total_Cost  Total_Units  Total_Orders  Num_Products  Avg_Margin  Avg_Profit_per_Unit  Profit_Efficiency  Revenue_Share_%  Profit_Share_%
Chocolate       77573.34      52353.28    25220.06        21953          4862             5       67.49                 2.39             0.6749            92.54           94.69
    Other        5906.25       2698.45     3207.80          701           177             3       39.01                 3.71             0.4569             7.05            4.88
    Sugar         347.84        239.64      108.20          101            28             7       58.75                 2.15             0.6889             0.41            0.43


### 2. Revenue vs Profit Imbalance

In [4]:
# Are divisions earning their fair share of profit relative to revenue?
# If Revenue_Share > Profit_Share → division is less efficient than average
# If Profit_Share > Revenue_Share → division punches above its weight

print("REVENUE vs PROFIT SHARE IMBALANCE\n")

for _, row in division_summary.iterrows():
    diff = row['Profit_Share_%'] - row['Revenue_Share_%']
    direction = "punches above weight" if diff > 0 else "underperforms on profit"
    print(f"  {row['Division']:<12} | Revenue: {row['Revenue_Share_%']:>6.2f}%  "
          f"| Profit: {row['Profit_Share_%']:>6.2f}%  "
          f"| Diff: {diff:>+.2f}%  | {direction}")

REVENUE vs PROFIT SHARE IMBALANCE

  Chocolate    | Revenue:  92.54%  | Profit:  94.69%  | Diff: +2.15%  | punches above weight
  Other        | Revenue:   7.05%  | Profit:   4.88%  | Diff: -2.17%  | underperforms on profit
  Sugar        | Revenue:   0.41%  | Profit:   0.43%  | Diff: +0.02%  | punches above weight


### 3. Division Performance Rating

In [5]:
# Rate each division across 3 dimensions
# Scale: score out of 10 relative to best performer

max_margin     = division_summary['Avg_Margin'].max()
max_efficiency = division_summary['Profit_Efficiency'].max()
max_profit     = division_summary['Total_Profit'].max()

division_summary['Margin_Score']     = (division_summary['Avg_Margin']      / max_margin     * 10).round(2)
division_summary['Efficiency_Score'] = (division_summary['Profit_Efficiency']/ max_efficiency * 10).round(2)
division_summary['Profit_Score']     = (division_summary['Total_Profit']     / max_profit     * 10).round(2)

division_summary['Overall_Score'] = (
    (division_summary['Margin_Score'] + 
     division_summary['Efficiency_Score'] + 
     division_summary['Profit_Score']) / 3
).round(2)

print("DIVISION PERFORMANCE SCORES (out of 10)")
print(division_summary[['Division', 'Margin_Score', 'Efficiency_Score', 
                         'Profit_Score', 'Overall_Score']].to_string(index=False))

DIVISION PERFORMANCE SCORES (out of 10)
 Division  Margin_Score  Efficiency_Score  Profit_Score  Overall_Score
Chocolate         10.00              9.80         10.00           9.93
    Other          5.78              6.63          0.52           4.31
    Sugar          8.70             10.00          0.05           6.25


### 4. Product Breakdown within Each Division

In [6]:
print("PRODUCTS WITHIN EACH DIVISION")

for division in division_summary['Division'].tolist():
    div_products = df[df['Division'] == division].groupby('Product Name').agg(
        Total_Revenue = ('Sales', 'sum'),
        Total_Profit  = ('Gross Profit', 'sum'),
        Avg_Margin    = ('Gross_Margin_%', 'mean')
    ).round(2).sort_values('Total_Profit', ascending=False).reset_index()

    div_revenue = div_products['Total_Revenue'].sum()
    div_profit  = div_products['Total_Profit'].sum()

    print(f"\n  {division.upper()} DIVISION:")
    print(f"  Total Revenue: ${div_revenue:,.2f} , Total Profit: ${div_profit:,.2f}")
    print(f"  {'Product':<35} {'Revenue':>10} {'Profit':>10} {'Margin':>8}")
    print(f"  {'-'*65}")
    for _, r in div_products.iterrows():
        print(f"  {r['Product Name']:<35} ${r['Total_Revenue']:>9,.2f} "
              f"${r['Total_Profit']:>9,.2f} {r['Avg_Margin']:>7.1f}%")

PRODUCTS WITHIN EACH DIVISION

  CHOCOLATE DIVISION:
  Total Revenue: $77,573.34 , Total Profit: $52,353.28
  Product                                Revenue     Profit   Margin
  -----------------------------------------------------------------
  Wonka Bar -Scrumdiddlyumptious      $16,344.00 $11,350.00    69.4%
  Wonka Bar - Triple Dazzle Caramel   $16,680.00 $10,897.60    65.3%
  Wonka Bar - Nutty Crunch Surprise   $14,452.09 $10,311.09    71.3%
  Wonka Bar - Milk Chocolate          $15,499.25 $10,062.59    64.9%
  Wonka Bar - Fudge Mallows           $14,598.00 $ 9,732.00    66.7%

  OTHER DIVISION:
  Total Revenue: $5,906.25 , Total Profit: $2,698.45
  Product                                Revenue     Profit   Margin
  -----------------------------------------------------------------
  Lickable Wallpaper                  $ 4,960.00 $ 2,480.00    50.0%
  Wonka Gum                           $   328.75 $   170.95    52.0%
  Kazookles                           $   617.50 $    47.50    

### 5. Quarterly Trend by Division

In [7]:
df['Quarter'] = df['Order Date'].dt.quarter

quarterly_division = df.groupby(['Division', 'Quarter']).agg(
    Total_Revenue = ('Sales', 'sum'),
    Total_Profit  = ('Gross Profit', 'sum'),
    Avg_Margin    = ('Gross_Margin_%', 'mean')
).round(2).reset_index()

print("QUARTERLY REVENUE BY DIVISION:")
pivot_revenue = quarterly_division.pivot(
    index='Quarter', columns='Division', values='Total_Revenue'
).round(2)
print(pivot_revenue.to_string())

print("\nQUARTERLY MARGIN BY DIVISION:")
pivot_margin = quarterly_division.pivot(
    index='Quarter', columns='Division', values='Avg_Margin'
).round(2)
print(pivot_margin.to_string())

QUARTERLY REVENUE BY DIVISION:
Division  Chocolate    Other   Sugar
Quarter                             
1          10787.75   641.00   37.24
2          16648.80   929.25  115.00
3          21310.90  1898.00   46.47
4          28825.89  2438.00  149.13

QUARTERLY MARGIN BY DIVISION:
Division  Chocolate  Other  Sugar
Quarter                          
1             67.55  44.22  61.69
2             67.48  32.20  58.61
3             67.52  37.53  62.82
4             67.46  40.68  56.50


### 6. Key Observations

In [8]:
print("KEY OBSERVATIONS:")

for _, row in division_summary.iterrows():
    print(f"\n  {row['Division']} Division")
    print(f"  Revenue Share  : {row['Revenue_Share_%']}%")
    print(f"  Profit Share   : {row['Profit_Share_%']}%")
    print(f"  Avg Margin     : {row['Avg_Margin']}%")
    print(f"  Profit/$ Rev   : ${row['Profit_Efficiency']:.4f}")
    print(f"  Overall Score  : {row['Overall_Score']} / 10")

# Flag the concentration risk
choc = division_summary[division_summary['Division'] == 'Chocolate'].iloc[0]
print(f"\n  CONCENTRATION RISK !!!")
print(f"  Chocolate division alone accounts for {choc['Revenue_Share_%']}% of revenue")
print(f"  and {choc['Profit_Share_%']}% of total profit.")
print(f"  Over-reliance on a single division is a structural business risk.")

KEY OBSERVATIONS:

  Chocolate Division
  Revenue Share  : 92.54%
  Profit Share   : 94.69%
  Avg Margin     : 67.49%
  Profit/$ Rev   : $0.6749
  Overall Score  : 9.93 / 10

  Other Division
  Revenue Share  : 7.05%
  Profit Share   : 4.88%
  Avg Margin     : 39.01%
  Profit/$ Rev   : $0.4569
  Overall Score  : 4.31 / 10

  Sugar Division
  Revenue Share  : 0.41%
  Profit Share   : 0.43%
  Avg Margin     : 58.75%
  Profit/$ Rev   : $0.6889
  Overall Score  : 6.25 / 10

  CONCENTRATION RISK !!!
  Chocolate division alone accounts for 92.54% of revenue
  and 94.69% of total profit.
  Over-reliance on a single division is a structural business risk.


### Save Report

In [9]:
division_report = {
    "division_summary"   : division_summary.to_dict(orient='records'),
    "quarterly_by_division": quarterly_division.to_dict(orient='records'),
    "key_flags": {
        "concentration_risk": (
            "Chocolate division contributes over 92% of revenue and 95% of profit. "
            "Sugar division is effectively negligible at 0.3% revenue share. "
            "This is a structural risk — any disruption to Chocolate impacts the entire business."
        ),
        "most_efficient_division": division_summary.iloc[0]['Division'],
        "least_efficient_division": division_summary.iloc[-1]['Division'],
        "top_overall_score": division_summary.sort_values(
            'Overall_Score', ascending=False).iloc[0]['Division']
    }
}

report_path = '../outputs/reports/division_analysis_report.json'
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(division_report, f, indent=4)

print(f"Division analysis report saved to: {report_path}")

Division analysis report saved to: ../outputs/reports/division_analysis_report.json
